# Lesson 02 - Erkundung des Microsoft Agent Framework

Das **Microsoft Agent Framework (MAF)** ist ein einheitliches Framework zum Erstellen von KI-Agenten. Es bietet eine klare, zusammensetzbare Architektur mit vier Kernbausteinen:

- **Client** – verbindet sich mit einem KI-Modell-Endpunkt und verwaltet die Kommunikation
- **Agent** – umschließt einen Client mit Anweisungen und Werkzeugdefinitionen
- **Tools** – erweitern die Fähigkeiten des Agenten mit benutzerdefinierten Funktionen, die das Modell aufrufen kann
- **Session** – verwaltet die Gesprächshistorie für mehrstufige Interaktionen

In dieser Lektion bauen wir einen **Reisebuchungsagenten**, der die Verfügbarkeit von Reisezielen mit diesen Konzepten prüft.


## Einrichtung


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Verständnis der Agent Framework Architektur

Das Microsoft Agent Framework folgt einer geschichteten Architektur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Ein `AzureAIProjectAgentProvider` verbindet sich mit einer Azure OpenAI-Bereitstellung. Er kümmert sich um Authentifizierung, Anforderungsformatierung und Antwortanalyse.
2. **Agent** – Vom Client über `provider.create_agent()` erstellt, kombiniert der Agent den Modellzugang mit Anweisungen (Systemprompt) und Werkzeugen.
3. **Werkzeuge** – Python-Funktionen, die mit `@tool` dekoriert sind und vom Agenten aufgerufen werden können, um Aktionen auszuführen oder Daten abzurufen.
4. **Sitzung** – Ein `AgentSession`-Objekt (erstellt über `agent.create_session()`), das den Gesprächsverlauf speichert und so einen mehrstufigen Dialog ermöglicht, bei dem sich der Agent an den vorherigen Kontext erinnert.

Lassen Sie uns jede Schicht Schritt für Schritt aufbauen.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hinzufügen von Tools mit dem @tool-Dekorator

Tools ermöglichen es Agenten, Aktionen über die reine Texterzeugung hinaus auszuführen. Der `@tool`-Dekorator verwandelt eine normale Python-Funktion in etwas, das der Agent aufrufen kann.

Wichtige Punkte:
- Verwenden Sie `Annotated[type, "Beschreibung"]`, damit das Modell jeden Parameter versteht.
- Der Docstring wird zur Tool-Beschreibung, die das Modell sieht.
- `approval_mode="never_require"` bedeutet, dass das Tool automatisch ohne Benutzerbestätigung ausgeführt wird.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Erstellen eines Agenten mit Werkzeugen

Nun kombinieren wir den Client, die Anweisungen und die Werkzeuge zu einem Agenten. Die `instructions` fungieren als Systemaufforderung – sie definieren die Persönlichkeit und das Verhalten des Agenten.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mehrstufige Gespräche mit Sessions

Eine `AgentSession` (erstellt über `agent.create_session()`) verfolgt alle Nachrichten in einem Gespräch. Indem dieselbe Session an jeden `agent.run()`-Aufruf übergeben wird, hat der Agent Zugriff auf den gesamten Gesprächsverlauf und kann auf frühere Nachrichten Bezug nehmen.

Wir übergeben `tools=[check_destination_availability]`, damit der Agent bei jedem Schritt unseren Verfügbarkeitsprüfer aufrufen kann.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Zusammenfassung

In dieser Lektion hast du die vier Säulen des Microsoft Agent Frameworks erkundet:

| Konzept | Was du gelernt hast |
|---------|---------------------|
| **Client** | `AzureAIProjectAgentProvider` verbindet sich mit Azure OpenAI über eine authentifizierungsbasierte Anmeldedaten-Authentifizierung |
| **Agent** | `provider.create_agent()` bündelt eine Modellverbindung mit Anweisungen und einem Namen |
| **Tools** | Der `@tool` Dekorator macht Python-Funktionen für den Agenten aufrufbar |
| **Sitzung** | `agent.create_session()` bewahrt den Gesprächsverlauf über mehrere Runden hinweg |

Diese Bausteine setzen sich zusammen, um Agenten zu schaffen, die natürliche Gespräche führen, externe Funktionen aufrufen und Kontext bewahren können — die Grundlage für fortgeschrittenere agentenbasierte Muster in späteren Lektionen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, kann die automatisierte Übersetzung Fehler oder Ungenauigkeiten enthalten. Das Originaldokument in seiner Ausgangssprache gilt als maßgebliche Quelle. Für kritische Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Nutzung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
